# 03 - Optimization Model
Use stochastic delay scenarios and robust MILP constraints to build a safer procurement plan.

## Step 1: Prepare inputs
Load data, retrain delay model, and append predicted delays.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Ensure project root (folder containing src/) is importable in notebook sessions.
project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_generation import default_component_demand
from src.delay_predictor import train_delay_model, predict_delays
from src.optimizer import generate_delay_scenarios, optimize_procurement

data_path = project_root / "data" / "supply_chain_dataset.csv"
df = pd.read_csv(data_path)
artifacts = train_delay_model(df)
df['predicted_delay_days'] = predict_delays(artifacts.pipeline, df)

## Step 2: Run optimization
Build demand, generate scenarios, and solve robust MILP with risk and carbon weights.

In [2]:
demand = default_component_demand(df)
scenarios = generate_delay_scenarios(
    predicted_delay_days=df['predicted_delay_days'].to_numpy(),
    residual_std=artifacts.metrics['residual_std'],
    n_scenarios=80,
    seed=42,
)
plan, summary = optimize_procurement(
    df=df,
    demand=demand,
    budget=2_000_000,
    deadline_days=28,
    delay_penalty=4.0,
    delay_scenarios=scenarios,
    risk_weight=0.2,
    carbon_weight=0.1,
    scenario_confidence=0.9,
    delay_variability_cap=3.0,
    solver_time_limit=10,
)
summary

{'status': 'Optimal',
 'procurement_cost': 1999605.4299823877,
 'fixed_cost': 31399.656916980908,
 'delay_cost': 15756.000118474683,
 'total_cost': 2046761.0870178433,
 'risk_score': 7.559813524295062,
 'carbon_score_kg': 12927.612649246545,
 'on_time_rate': 1.0,
 'budget': 2000000.0,
 'deadline_days': 28.0,
 'risk_weight': 0.2,
 'carbon_weight': 0.1,
 'scenario_count': 80,
 'supplier_reliability_variance': 0.010505482938063892}

## Step 3: Preview plan
Show the top selected procurement rows.

In [3]:
plan.head()

,supplier_id,component,quantity,unit_cost,fixed_penalty,predicted_delay_days,expected_delay_days,worst_delay_days,lead_time_days,arrival_days,worst_arrival_days,supplier_risk_coeff
0,S02,Cryogenic propellant tank,72,1489.874784,1072.692914,3.016948,3.084759,4.816178,5.158463,8.243222,9.974640,0.007883
1,S06,Cryogenic propellant tank,16,2054.445141,1135.053124,4.506157,4.691961,7.068052,4.545040,9.237001,11.613092,0.013300
2,S06,Grid fin assembly,90,1234.542720,625.867410,3.198966,3.339067,5.781060,9.249144,12.588211,15.030205,0.013300
3,S07,Merlin 1D turbopump,90,1595.910275,828.853479,5.177238,4.676417,7.640101,8.197180,12.873597,15.837281,0.005609
4,S07,Raptor injector,81,1544.859879,3973.721411,4.200629,4.065433,6.371397,3.060114,7.125547,9.431511,0.005609
